# CÂU 6 – PHÂN TÍCH DISCOUNT TRONG E-COMMERCE

**Mục tiêu:** phân tích mối liên hệ giữa `discount_pct` với số lượng sản phẩm (`quantity`) và giá trị giao dịch (`total_amount_usd`).

Notebook gồm: đọc dữ liệu, làm sạch, chia nhóm Discount, thống kê mô tả, kiểm định Spearman và Kruskal–Wallis, vẽ Box Plot và xuất kết quả.

**Cách dùng:** đặt notebook tại thư mục gốc `Ecommerce_analysis`, cùng cấp với `data_processed`. Dataset cần có ở `data_processed/orders_cleaned.csv`.

## 1. Import thư viện
Nếu thiếu thư viện, bỏ dấu `#` ở dòng pip install rồi chạy cell.

In [22]:
%pip install pandas numpy scipy matplotlib plotly

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
from pathlib import Path

print("Thư mục hiện tại:", Path.cwd())
print("Có thư mục data_processed không:",
      (Path.cwd() / "data_processed").exists())

Thư mục hiện tại: c:\Users\DELL\Downloads\Ecommerce_analysis
Có thư mục data_processed không: True


## 2. Tìm và đọc dữ liệu

In [24]:
DATA_RELATIVE = Path("data_processed") / "orders_cleaned.csv"
OUTPUT_DIR = Path("reports") / "cau6_discount"

search_dirs = [Path.cwd(), *Path.cwd().parents]
DATA_FILE = next(
    (folder / DATA_RELATIVE for folder in search_dirs
     if (folder / DATA_RELATIVE).exists()),
    None
)

if DATA_FILE is None:
    raise FileNotFoundError(
        "Không tìm thấy data_processed/orders_cleaned.csv. "
        "Hãy đặt notebook ở thư mục gốc Ecommerce_analysis."
    )

print("Dataset:", DATA_FILE.resolve())
df_raw = pd.read_csv(DATA_FILE)
print("Kích thước dữ liệu:", df_raw.shape)
display(df_raw.head())


Dataset: C:\Users\DELL\Downloads\Ecommerce_analysis\data_processed\orders_cleaned.csv
Kích thước dữ liệu: (235, 29)


,order_id,customer_id,order_date,year,month,quarter,day_of_week,product_name,category,unit_price_usd,quantity,subtotal_usd,discount_pct,discount_amount_usd,shipping_fee_usd,tax_pct,tax_amount_usd,total_amount_usd,payment_method,device_used,delivery_days,delivery_date,order_status,returned,customer_rating,session_duration_minutes,pages_viewed_before_purchase,is_repeat_customer,rating_provided
0,O000001,C07108,2020-08-27,2020,8,Q3,Thursday,Tire Inflator,Automotive,62.91,1,62.91,0,0.00,0.00,18,11.32,74.23,Credit Card,Desktop,3,2020-08-30,Delivered,0,NaN,14.40,1,1,0
1,O000002,C03487,2024-04-11,2024,4,Q2,Thursday,Stud Earrings Gold,Jewelry & Accessories,18.44,1,18.44,0,0.00,3.99,10,1.84,24.27,Credit Card,Mobile,2,2024-04-13,Delivered,0,NaN,9.00,4,0,0
2,O000003,C03062,2023-06-25,2023,6,Q2,Sunday,Pen Set Premium,Office Supplies,109.79,1,109.79,20,21.96,0.00,8,7.03,94.86,PayPal,Desktop,3,2023-06-28,Delivered,0,4.00,3.20,17,0,1
3,O000004,C00888,2020-08-16,2020,8,Q3,Sunday,Smart Watch Series 5,Electronics,87.21,1,87.21,50,43.60,0.00,8,3.49,47.10,Credit Card,Tablet,2,2020-08-18,Returned,1,NaN,32.20,3,1,0
4,O000005,C01674,2020-04-24,2020,4,Q2,Friday,Stud Earrings Gold,Jewelry & Accessories,94.30,2,188.60,0,0.00,9.99,5,9.43,208.02,Debit Card,Mobile,5,2020-04-29,Delivered,0,2.80,54.70,2,1,1


## 3. Kiểm tra các cột cần thiết
Ba cột theo notebook phân tích hiện tại là `discount_pct`, `quantity`, `total_amount_usd`.

In [25]:
required = ["discount_pct", "quantity", "total_amount_usd"]
missing = [c for c in required if c not in df_raw.columns]
if missing:
    raise ValueError(f"Dataset thiếu cột: {missing}")

display(df_raw[required].head())
display(df_raw[required].describe())
print("Giá trị thiếu:")
display(df_raw[required].isna().sum())


,discount_pct,quantity,total_amount_usd
0,0,1,74.23
1,0,1,24.27
2,20,1,94.86
3,50,1,47.10
4,0,2,208.02


,discount_pct,quantity,total_amount_usd
count,235.00,235.00,235.00
mean,5.36,1.61,118.66
std,9.64,0.97,129.36
min,0.00,1.00,11.03
25%,0.00,1.00,47.16
50%,0.00,1.00,83.32
75%,10.00,2.00,138.47
max,50.00,5.00,901.63


Giá trị thiếu:


discount_pct        0
quantity            0
total_amount_usd    0
dtype: int64

## 4. Làm sạch dữ liệu
Chuyển cột về kiểu số, loại dòng thiếu dữ liệu và giá trị không hợp lệ. Discount nằm trong khoảng 0–100; Quantity và Amount không âm.

In [26]:
df = df_raw.copy()
for col in required:
    df[col] = pd.to_numeric(df[col], errors="coerce")

n_before = len(df)
df = df.dropna(subset=required).copy()
df = df[
    df["discount_pct"].between(0, 100)
    & (df["quantity"] >= 0)
    & (df["total_amount_usd"] >= 0)
].copy()

print(f"Số dòng ban đầu: {n_before:,}")
print(f"Số dòng hợp lệ: {len(df):,}")
print(f"Số dòng bị loại: {n_before - len(df):,}")
if df.empty:
    raise ValueError("Không còn dòng dữ liệu hợp lệ.")


Số dòng ban đầu: 235
Số dòng hợp lệ: 235
Số dòng bị loại: 0


## 5. Chia Discount thành 5 nhóm
Nhóm 0%, 1–10%, 11–20%, 21–30% và trên 30%.

In [27]:
discount_bins = [-0.001, 0, 10, 20, 30, 100]
discount_labels = [
    "0% (Không giảm)", "1–10%", "11–20%", "21–30%", "Trên 30%"
]

df["discount_group"] = pd.cut(
    df["discount_pct"],
    bins=discount_bins,
    labels=discount_labels,
    include_lowest=True,
    right=True
)

display(
    df["discount_group"]
    .value_counts(sort=False, dropna=False)
    .rename("Số giao dịch")
    .to_frame()
)


,Số giao dịch
discount_group,
0% (Không giảm),161
1–10%,26
11–20%,32
21–30%,11
Trên 30%,5


## 6. Thống kê mô tả theo nhóm Discount

In [28]:
summary = (
    df.groupby("discount_group", observed=False)
    .agg(
        so_giao_dich=("quantity", "size"),
        quantity_trung_binh=("quantity", "mean"),
        quantity_trung_vi=("quantity", "median"),
        amount_trung_binh=("total_amount_usd", "mean"),
        amount_trung_vi=("total_amount_usd", "median"),
        tong_amount=("total_amount_usd", "sum"),
        discount_trung_binh=("discount_pct", "mean")
    )
)
summary["ty_trong_giao_dich_pct"] = (
    summary["so_giao_dich"] / summary["so_giao_dich"].sum() * 100
)
summary = summary.round(2)
display(summary)


,so_giao_dich,quantity_trung_binh,quantity_trung_vi,amount_trung_binh,amount_trung_vi,tong_amount,discount_trung_binh,ty_trong_giao_dich_pct
discount_group,,,,,,,,
0% (Không giảm),161,1.60,1.00,122.20,83.32,"19,674.10",0.00,68.51
1–10%,26,1.85,1.00,136.25,113.38,"3,542.41",7.69,11.06
11–20%,32,1.72,1.00,113.34,84.35,"3,626.87",17.19,13.62
21–30%,11,1.18,1.00,77.14,46.69,848.51,27.27,4.68
Trên 30%,5,1.00,1.00,38.75,35.61,193.73,42.00,2.13


## 7. Kiểm định tương quan Spearman

Spearman đo chiều hướng và mức độ liên hệ đơn điệu:
- rho dương: hai biến có xu hướng cùng chiều.
- rho âm: hai biến có xu hướng ngược chiều.
- rho gần 0: liên hệ đơn điệu yếu.

Dùng p-value với mức ý nghĩa 0.05. Tương quan không chứng minh quan hệ nhân quả.

In [29]:
spearman_quantity = stats.spearmanr(df["discount_pct"], df["quantity"])
spearman_amount = stats.spearmanr(df["discount_pct"], df["total_amount_usd"])

spearman_results = pd.DataFrame([
    {"Cặp biến": "Discount – Quantity",
     "Spearman rho": spearman_quantity.statistic,
     "p-value": spearman_quantity.pvalue},
    {"Cặp biến": "Discount – Amount",
     "Spearman rho": spearman_amount.statistic,
     "p-value": spearman_amount.pvalue}
])
display(spearman_results.round(6))


,Cặp biến,Spearman rho,p-value
0,Discount – Quantity,-0.01,0.85
1,Discount – Amount,-0.06,0.33


## 8. Kiểm định Kruskal–Wallis

Kiểm tra phân bố Quantity và Amount có khác biệt giữa các nhóm Discount hay không. Nếu p < 0.05, có bằng chứng rằng ít nhất một nhóm khác biệt; kiểm định này không cho biết cụ thể nhóm nào khác nhóm nào.

In [30]:
quantity_groups = [
    g["quantity"].to_numpy()
    for _, g in df.groupby("discount_group", observed=True) if len(g)
]
amount_groups = [
    g["total_amount_usd"].to_numpy()
    for _, g in df.groupby("discount_group", observed=True) if len(g)
]

kw_quantity = stats.kruskal(*quantity_groups) if len(quantity_groups) >= 2 else None
kw_amount = stats.kruskal(*amount_groups) if len(amount_groups) >= 2 else None

kruskal_results = pd.DataFrame([
    {"Biến": "Quantity", "H statistic": kw_quantity.statistic if kw_quantity else np.nan,
     "p-value": kw_quantity.pvalue if kw_quantity else np.nan},
    {"Biến": "Amount", "H statistic": kw_amount.statistic if kw_amount else np.nan,
     "p-value": kw_amount.pvalue if kw_amount else np.nan}
])
display(kruskal_results.round(6))


,Biến,H statistic,p-value
0,Quantity,6.62,0.16
1,Amount,11.10,0.03


## 9. Box Plot – Discount và Quantity
Đường giữa hộp là trung vị; mép hộp là Q1 và Q3; các điểm ngoài râu thường được xem là ngoại lệ.

In [31]:
fig_quantity = px.box(
    df, x="discount_group", y="quantity",
    category_orders={"discount_group": discount_labels},
    points="outliers",
    title="Phân bố số lượng sản phẩm theo nhóm Discount",
    labels={"discount_group": "Nhóm chiết khấu", "quantity": "Số lượng sản phẩm"}
)
fig_quantity.update_layout(showlegend=False, template="plotly_white")
fig_quantity.show()


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## 10. Box Plot – Discount và Amount

In [ ]:
fig_amount = px.box(
    df, x="discount_group", y="total_amount_usd",
    category_orders={"discount_group": discount_labels},
    points="outliers",
    title="Phân bố giá trị giao dịch theo nhóm Discount",
    labels={"discount_group": "Nhóm chiết khấu",
            "total_amount_usd": "Giá trị giao dịch (USD)"}
)
fig_amount.update_layout(showlegend=False, template="plotly_white")
fig_amount.show()


## 11. Nhận xét tự động từ kết quả thực tế
Không kết luận quan hệ nhân quả chỉ từ tương quan hoặc kiểm định.

In [ ]:
def spearman_comment(name, result):
    rho, p = result.statistic, result.pvalue
    direction = "cùng chiều" if rho > 0 else "ngược chiều" if rho < 0 else "không thể hiện chiều hướng"
    significance = "có ý nghĩa thống kê" if p < 0.05 else "chưa có ý nghĩa thống kê"
    return f"{name}: rho={rho:.4f}, p={p:.6g}; xu hướng {direction}, {significance}."

print(spearman_comment("Discount – Quantity", spearman_quantity))
print(spearman_comment("Discount – Amount", spearman_amount))

for name, result in [("Quantity", kw_quantity), ("Amount", kw_amount)]:
    if result is not None:
        conclusion = "có khác biệt phân bố giữa ít nhất hai nhóm" if result.pvalue < 0.05 else "chưa có bằng chứng khác biệt phân bố giữa các nhóm"
        print(f"Kruskal–Wallis {name}: H={result.statistic:.4f}, p={result.pvalue:.6g}; {conclusion}.")

print("Lưu ý: kết quả thể hiện mối liên hệ/khác biệt, không tự chứng minh Discount là nguyên nhân.")


## 12. Lưu bảng, biểu đồ và nhận xét
Kết quả được lưu vào `reports/cau6_discount`. PNG dùng để xem trực tiếp trên GitHub; HTML là biểu đồ Plotly tương tác.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary.to_csv(OUTPUT_DIR / "discount_summary.csv", encoding="utf-8-sig")
fig_quantity.write_html(OUTPUT_DIR / "boxplot_quantity.html")
fig_amount.write_html(OUTPUT_DIR / "boxplot_amount.html")

# Lưu PNG bằng Matplotlib
fig, ax = plt.subplots(figsize=(10, 6))
q_data = [df.loc[df["discount_group"] == label, "quantity"].dropna()
          for label in discount_labels]
ax.boxplot(q_data, labels=discount_labels, showfliers=True)
ax.set_title("Phân bố số lượng sản phẩm theo nhóm Discount")
ax.set_xlabel("Nhóm chiết khấu")
ax.set_ylabel("Số lượng sản phẩm")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=15)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "boxplot_quantity.png", dpi=180)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
a_data = [df.loc[df["discount_group"] == label, "total_amount_usd"].dropna()
          for label in discount_labels]
ax.boxplot(a_data, labels=discount_labels, showfliers=True)
ax.set_title("Phân bố giá trị giao dịch theo nhóm Discount")
ax.set_xlabel("Nhóm chiết khấu")
ax.set_ylabel("Giá trị giao dịch (USD)")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=15)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "boxplot_amount.png", dpi=180)
plt.show()
plt.close(fig)

report = [
    "KET QUA PHAN TICH CAU 6 - DISCOUNT",
    f"Tong so giao dich hop le: {len(df):,}",
    spearman_comment("Discount va Quantity", spearman_quantity),
    spearman_comment("Discount va Amount", spearman_amount),
    f"Kruskal-Wallis Quantity: H={kw_quantity.statistic:.4f}, p={kw_quantity.pvalue:.6g}" if kw_quantity else "Kruskal-Wallis Quantity: khong du nhom",
    f"Kruskal-Wallis Amount: H={kw_amount.statistic:.4f}, p={kw_amount.pvalue:.6g}" if kw_amount else "Kruskal-Wallis Amount: khong du nhom",
    "Luu y: ket qua khong tu dong chung minh quan he nhan qua."
]
(OUTPUT_DIR / "cau6_nhan_xet.txt").write_text("\n".join(report), encoding="utf-8")

print("Đã lưu kết quả tại:", OUTPUT_DIR.resolve())
print("Các file:", sorted(p.name for p in OUTPUT_DIR.iterdir()))
